# Pinoy Recipe RAG: setup and end-to-end test

Progress check notebook. Pipeline: query -> embed -> retrieve top-k recipes -> prompt -> local LLM (Ollama).

In [1]:
%pip install -q ollama sentence-transformers datasets pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Config

In [2]:
import numpy as np
import pandas as pd
import ollama
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

LLM_MODEL = "llama3.2:3b"                      # exact Ollama tag
EMBED_MODEL = "intfloat/multilingual-e5-small"  # small and fast; switch to -base later
TOP_K = 3
SEED = 42

## Load the dataset

In [3]:
ds = load_dataset("joackimagno/FILIPINO_RECIPES_2K")
split = list(ds.keys())[0]          # usually "train"
df = ds[split].to_pandas()

print("split:", split, "| rows:", len(df))
print("columns:", list(df.columns))
df.head(3)

README.md:   0%|          | 0.00/559 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.12MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1988 [00:00<?, ? examples/s]

split: train | rows: 1988
columns: ['recipe_name', 'prep_time', 'cook_time', 'total_time', 'servings', 'ingredients', 'instructions', 'ingredient_names']


,recipe_name,prep_time,cook_time,total_time,servings,ingredients,instructions,ingredient_names
0,Pork Hock and Mung Beans with Vermicelli (Gini...,10 minutes,1 hour 10 minutes,1 hour 20 minutes,5.0,3 lbs. pork hocks cleaned | 1 1/2 cups mung be...,Boil the pork hock in 4 cups of water for 10 m...,pork | mung bean | sotanghon | beef powder | t...
1,Ginataang Kalabasa,15 minutes,50 minutes,1 hour 5 minutes,6.0,3 lbs. kalabasa cubed | 18 pieces string beans...,Heat the wok and then sear the pork until enou...,kalabasa | sitaw | pork | coconut milk | thumb...
2,Sinigang Sa Miso,30 minutes,40 minutes,1 hour 10 minutes,4.0,2 lbs. bangus milkfish cleaned and sliced | 25...,Rub salt all over the fish. Let it stay for 10...,bangus milkfish | kangkong | sitaw | okra | lo...


Check the printed columns. The code below expects `recipe_name`, `ingredients`, `instructions`, and `ingredient_names`.
If a name is different, change it in the next cell.

In [ ]:
COL_NAME = "recipe_name"
COL_ING = "ingredients"
COL_INSTR = "instructions"
COL_ING_NAMES = "ingredient_names"

for c in [COL_NAME, COL_ING, COL_INSTR, COL_ING_NAMES]:
    assert c in df.columns, f"Column {c} not found. Available: {list(df.columns)}"

# Text that gets embedded for each recipe
df["doc_text"] = df[COL_NAME].astype(str) + ". Ingredients: " + df[COL_ING_NAMES].astype(str)
df["doc_text"].head(3).tolist()

## Embed the recipes (dense retrieval)

In [ ]:
embedder = SentenceTransformer(EMBED_MODEL)

# e5 models expect these prefixes
passages = ["passage: " + t for t in df["doc_text"].tolist()]
recipe_vecs = embedder.encode(passages, normalize_embeddings=True, show_progress_bar=True)
print(recipe_vecs.shape)

## Retriever

In [ ]:
def retrieve(query, k=TOP_K):
    q_vec = embedder.encode(["query: " + query], normalize_embeddings=True)[0]
    scores = recipe_vecs @ q_vec               # cosine similarity (vectors are normalized)
    top_idx = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in top_idx]

# quick check
for i, s in retrieve("may sitaw, bawang, at manok ako"):
    print(round(s, 3), "|", df.loc[i, COL_NAME])

## Basic RAG prompt template

In [ ]:
SYSTEM_PROMPT = (
    "You are a Filipino recipe assistant. "
    "Use ONLY the recipes provided in the context. "
    "Do not invent ingredients, quantities, cooking times, or steps that are not in the context. "
    "Reply in the same language as the user's message (English, Tagalog, or Taglish)."
)

USER_TEMPLATE = """The user has these ingredients: {pantry}

Context recipes:
{context}

Tasks:
1. Recommend the best matching dish from the context and briefly explain why it fits.
2. List the matched ingredients (the user has them) and the missing ingredients.
3. For each missing ingredient, suggest a substitute only if it is reasonable; otherwise say there is none.
4. Rewrite the cooking instructions as short numbered steps. Keep all quantities and times."""

def format_context(indices, max_chars=1200):
    blocks = []
    for n, i in enumerate(indices, start=1):
        row = df.loc[i]
        blocks.append(
            f"[Recipe {n}] {row[COL_NAME]}\n"
            f"Ingredients: {row[COL_ING]}\n"
            f"Instructions: {str(row[COL_INSTR])[:max_chars]}"
        )
    return "\n\n".join(blocks)

def build_messages(pantry, indices):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(pantry=pantry, context=format_context(indices))},
    ]

## Generation through Ollama

In [ ]:
def generate(messages):
    response = ollama.chat(
        model=LLM_MODEL,
        messages=messages,
        options={"temperature": 0, "seed": SEED},
    )
    return response["message"]["content"]

## End-to-end test (one query)

In [ ]:
query = "may sitaw, bawang, at manok ako"

hits = retrieve(query, k=TOP_K)
print("Retrieved recipes:")
for i, s in hits:
    print(f"  {s:.3f} | {df.loc[i, COL_NAME]}")

answer = generate(build_messages(query, [i for i, _ in hits]))
print("\n--- LLM answer ---\n")
print(answer)